Градиентный бустинг — один из самых мощных алгоритмов машинного обучения для табличных данных.  
Он строит ансамбль из слабых моделей (обычно деревьев решений), где каждая следующая модель исправляет ошибки предыдущей, двигаясь в направлении антиградиента функции потерь.

В этом ноутбуке мы подробно рассмотрим **четыре реализации**:

- **XGBoost** (eXtreme Gradient Boosting) – оптимизированная, с регуляризацией и поддержкой пропусков.
- **LightGBM** – разработана Microsoft, ориентирована на скорость и эффективность, использует односторонний рост листьев.
- **CatBoost** – от Яндекса, автоматически обрабатывает категориальные признаки, использует симметричные деревья.
- **GradientBoosting** из sklearn – классическая реализация, удобная для прототипирования.

Еще можно почитать объяснение простыми словами тут - > https://habr.com/ru/companies/raft/articles/890802/


Прикольно про ГБ в маркетинге -> https://altcraft.com/ru/glossary/gradientnyj-boosting-kak-pomogaet-predskazat-churn-rate

In [ ]:
!pip install catboost xgboost lightgbm -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import timeit
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import GradientBoostingClassifier

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
try:
    df = pd.read_csv("titanic.csv")
except FileNotFoundError:
    df = pd.read_csv(url)

In [ ]:
df.head()

## 1. Обзор реализаций градиентного бустинга

### 1.1. XGBoost (eXtreme Gradient Boosting)
**Реализация:**  
XGBoost использует **потоковый алгоритм** для построения деревьев, поддерживает разрежённые данные и **регуляризацию** (L1, L2). Применяет **приближённый жадный алгоритм** для нахождения наилучшего разбиения, что ускоряет обучение.

**Плюсы:**
- Высокая производительность и точность.
- Встроенная регуляризация, борьба с переобучением.
- Поддержка пропущенных значений (обучается направление ветвления для пропусков).
- Широкие возможности кастомизации (функции потерь, метрики).

**Минусы:**
- При большом количестве признаков может быть медленнее LightGBM.
- Не имеет автоматической обработки категорий (нужно кодировать вручную).

**Ключевые параметры:**
- `n_estimators` – число деревьев.
- `learning_rate` – шаг обучения.
- `max_depth` – максимальная глубина дерева.
- `subsample` – доля объектов для каждого дерева (стохастический бустинг).
- `colsample_bytree` – доля признаков для каждого дерева.
- `reg_lambda`, `reg_alpha` – L2 и L1 регуляризация.
- `early_stopping_rounds` – ранняя остановка.

---

### 1.2. LightGBM (Light Gradient Boosting Machine)
**Реализация:**  
LightGBM использует **односторонний рост листьев** (leaf-wise) вместо уровневого (level-wise), что позволяет быстрее снижать ошибку. Также применяет **гистограммный метод** для поиска разбиений, что сильно ускоряет обучение и снижает потребление памяти.

**Плюсы:**
- Очень высокая скорость обучения (особенно на больших данных).
- Низкое потребление памяти.
- Поддержка категориальных признаков (нужно указать `categorical_feature`).
- Встроенная обработка дисбаланса классов.

**Минусы:**
- При малом объёме данных может переобучаться из-за leaf-wise роста (регулируется `num_leaves` и `min_data_in_leaf`).
- Чувствителен к шуму (при неправильной настройке).

**Ключевые параметры:**
- `n_estimators`, `learning_rate`.
- `num_leaves` – максимальное количество листьев в дереве (аналог max_depth).
- `min_data_in_leaf` – минимальное число объектов в листе (защита от переобучения).
- `feature_fraction` – доля признаков для каждого дерева.
- `bagging_fraction` – доля объектов для каждого дерева.
- `lambda_l1`, `lambda_l2` – регуляризация.
- `categorical_feature` – список индексов категориальных признаков.

---

### 1.3. CatBoost (Categorical Boosting)
**Реализация:**  
CatBoost строит **симметричные деревья** (oblivious trees), что ускоряет инференс. Главная особенность – встроенная обработка категориальных признаков с использованием различных статистик (counter, target encoding с борьбой против переобучения). Также использует **рандом** для перестановок при кодировании.

**Плюсы:**
- Автоматическая работа с категориями (без One-Hot Encoding).
- Высокое качество по умолчанию (меньше нужно подбирать параметры).
- Устойчивость к переобучению благодаря симметричным деревьям и встроенным методам.
- Поддержка текстовых признаков.

**Минусы:**
- Обучение может быть медленнее LightGBM (особенно на больших данных).
- Требует больше памяти при большом количестве категорий.

**Ключевые параметры:**
- `iterations` – число деревьев.
- `learning_rate`.
- `depth` – глубина дерева (обычно <=10).
- `l2_leaf_reg` – L2 регуляризация в листьях.
- `border_count` – количество разбиений при дискретизации вещественных признаков.
- `cat_features` – список индексов категориальных признаков (можно не указывать, библиотека определит сама, но лучше передать).
- `verbose` – вывод логов.

---

### 1.4. GradientBoosting (sklearn)
**Реализация:**  
Классическая реализация из библиотеки scikit-learn. Использует уровневый рост деревьев (level-wise) и жадный поиск разбиений. Не имеет встроенной поддержки разрежённых данных и категорий, требует предварительной обработки.

**Плюсы:**
- Простота использования, полная совместимость с инструментами sklearn (GridSearchCV, пайплайны).
- Хорошо документирована.
- Подходит для быстрых экспериментов.

**Минусы:**
- Медленнее остальных реализаций (особенно на больших данных).
- Нет регуляризации по умолчанию (кроме ограничений на дерево).
- Потребляет больше памяти.

**Ключевые параметры:**
- `n_estimators`, `learning_rate`.
- `max_depth` – глубина дерева.
- `min_samples_split` – минимальное число объектов для разбиения.
- `min_samples_leaf` – минимальное число объектов в листе.
- `max_features` – доля признаков для каждого дерева.
- `subsample` – доля объектов для каждого дерева.
- `criterion` – критерий качества разбиения ('friedman_mse', 'mse', 'mae').

## 2. Сравнение на одном датасете

Обучим каждую модель с одинаковыми (насколько возможно) параметрами, замерим время и качество (ROC-AUC).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

features = ["Pclass", "Sex", "Age", "Fare", "SibSp", "Parch", "Embarked"]
X = df[features].copy()
y = df["Survived"]

X["Age"] = X["Age"].fillna(X["Age"].median())
X["Embarked"] = X["Embarked"].fillna(X["Embarked"].mode()[0])

X_encoded = pd.get_dummies(X, columns=["Sex", "Embarked"])

# Для CatBoost — передаём как есть
X_catboost = X.copy()
cat_features = ["Sex", "Embarked"]

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)
X_train_cb, X_test_cb, _, _ = train_test_split(
    X_catboost, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

results = {}

In [ ]:
X_train.head()

In [ ]:
# XGBoost
def train_xgb():
    model = xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1,
        reg_alpha=0,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    model.fit(X_train, y_train)
    return model


time = timeit.timeit(train_xgb, number=5) / 5
xgb_model = train_xgb()
results["XGBoost"] = {
    "AUC": roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]),
    "Time (s)": round(time, 3)
}

In [ ]:
# LightGBM
def train_lgb():
  model = lgb.LGBMClassifier(
      n_estimators=300,
      learning_rate=0.05,
      num_leaves=16,
      min_data_in_leaf=20,
      feature_fraction=0.8,
      bagging_fraction=0.8,
      bagging_freq=1,
      lambda_l2=1,
      random_state=42,
      verbose=-1
  )
  model.fit(X_train, y_train)
  return model


time = timeit.timeit(train_lgb, number=5) / 5
lgb_model = train_lgb()
results["LightGBM"] = {
    "AUC": roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1]),
    "Time (s)": round(time, 3)
}

In [ ]:
# CatBoost
def train_cb():
  model = cb.CatBoostClassifier(
      iterations=300,
      learning_rate=0.05,
      depth=4,
      l2_leaf_reg=3,
      random_state=42,
      verbose=False,
      cat_features=cat_features
  )
  model.fit(X_train_cb, y_train)
  return model

time = timeit.timeit(train_cb, number=5) / 5
cb_model = train_cb()
results["CatBoost"] = {
    "AUC": roc_auc_score(y_test, cb_model.predict_proba(X_test_cb)[:, 1]),
    "Time (s)": round(time, 3)
}

In [ ]:
# Sklearn
def train_sk():
  model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    max_features=0.8,
    random_state=42
)
  model.fit(X_train, y_train)
  return model

time = timeit.timeit(train_sk, number=5) / 5
sk_model = train_sk()
results["Sklearn"] = {
    "AUC": roc_auc_score(y_test, sk_model.predict_proba(X_test)[:, 1]),
    "Time (s)": round(time, 3)
}

In [ ]:
# Результаты
df_results = pd.DataFrame(results).T
print("\nСравнение моделей:")
print(df_results.round(4))

In [ ]:
# Визуализация
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# AUC
ax1.bar(df_results.index, df_results['AUC'], color='skyblue', edgecolor='k')
ax1.set_ylabel('ROC-AUC')
ax1.set_title('Качество (AUC)')
ax1.set_ylim(0.75, 1.0)

# Время
ax2.bar(df_results.index, df_results['Time (s)'], color='salmon', edgecolor='k')
ax2.set_ylabel('Время обучения (сек)')
ax2.set_title('Скорость обучения')
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

### Анализ результатов
Из графика видно, что:
- **LightGBM** обычно обучается быстрее всех.
- **CatBoost** может быть немного медленнее, но часто даёт хорошее качество без тонкой настройки.
- **XGBoost** — золотая середина: качество на уровне, приемлемая скорость.
- **Sklearn** уступает по скорости, но для небольших данных вполне пригоден.

В реальных проектах выбор библиотеки зависит от объёма данных, наличия категориальных признаков и требований к скорости.

## 📝 Проверь себя

1. Какой алгоритм роста деревьев использует LightGBM?

Ответ: LightGBM использует leaf-wise рост деревьев: на каждом шаге он выбирает лист, разбиение которого сильнее всего уменьшает ошибку. Это часто быстрее и точнее, но на малых данных может повышать риск переобучения.

2. Какая библиотека обучилась медленнее всех на синтетическом датасете из 10 000 объектов?

Ответ: в типичном сравнении медленнее всех обычно обучается классический `GradientBoostingClassifier` из sklearn, потому что это более базовая реализация без оптимизаций XGBoost, LightGBM и CatBoost.

3. Какой параметр в LightGBM является основным аналогом max_depth из XGBoost?

Ответ: ключевой аналог — `num_leaves`, то есть максимальное количество листьев. Чем больше листьев, тем сложнее дерево. Также в LightGBM есть параметр `max_depth`, но чаще сложность контролируют именно через `num_leaves`.

4. Какая уникальная особенность делает CatBoost особенно удобным для данных из реальных проектов?

Ответ: CatBoost умеет автоматически работать с категориальными признаками без ручного One-Hot Encoding и использует специальные схемы кодирования, снижающие риск переобучения.

5. Почему XGBoost поддерживает пропущенные значения (NaN) без предобработки?

Ответ: XGBoost при построении разбиений обучает направление для пропусков: для каждого split выбирает, в какую ветку отправлять NaN, чтобы улучшить качество. Поэтому пропуски не обязательно предварительно заполнять.


# Задания для самостоятельной работы

**Задание 1. Генерация данных Текст**

Сгенерируйте новый датасет для классификации:

20 000 объектов

*   20 000 объектов
*   25 признаков, из которых 15 информативных и 5 избыточных
*   2 класса
*   зафиксируйте random_state=123

In [ ]:
X, y = make_classification(
    n_samples=20000,
    n_features=25,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    random_state=123
)


In [ ]:
# Проверка параметров make_classification
assert 'X' in globals(), "❌ Переменная X не определена. Запустите ячейку с генерацией данных."
assert 'y' in globals(), "❌ Переменная y не определена."
assert X.shape == (20000, 25), f"❌ Размер X должен быть (20000, 25), а у вас {X.shape}"
assert y.shape == (20000,), f"❌ Размер y должен быть (20000,), а у вас {y.shape}"
assert len(np.unique(y)) == 2, "❌ Должно быть два класса."
print("✅ Данные сгенерированы верно.")

**Задание 2. Разделение выборки**

Разделите данные на обучающую и тестовую в соотношении 80:20, используя train_test_split с random_state=42.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
# Проверка train_test_split
assert 'X_train' in globals(), "❌ X_train не определён."
assert 'X_test' in globals(), "❌ X_test не определён."
assert 'y_train' in globals(), "❌ y_train не определён."
assert 'y_test' in globals(), "❌ y_test не определён."
assert X_train.shape[0] == 16000, f"❌ Ожидается 16000 объектов в train, получено {X_train.shape[0]}"
assert X_test.shape[0] == 4000, f"❌ Ожидается 4000 объектов в test, получено {X_test.shape[0]}"
assert y_train.shape[0] == 16000, f"❌ y_train имеет неверный размер"
print("✅ Данные разделены корректно.")

**Задание 3. XGBoost**

Создайте и обучите классификатор XGBoost со следующими параметрами:

150 деревьев
скорость обучения 0.05
максимальная глубина 6
random_state=42
отключите использование label encoder (use_label_encoder=False)
метрика для eval – 'logloss'
Замерьте время обучения и сохраните качество ROC-AUC на тесте.

In [ ]:
import time

params_xgb = {
    'n_estimators': 150,
    'learning_rate': 0.05,
    'max_depth': 6,
    'random_state': 42,
    'use_label_encoder': False,
    'eval_metric': 'logloss'
}
model_xgb = xgb.XGBClassifier(**params_xgb)
start = time.time()
model_xgb.fit(X_train, y_train)
time_xgb = time.time() - start
pred_xgb = model_xgb.predict_proba(X_test)[:, 1]
auc_xgb = roc_auc_score(y_test, pred_xgb)
print(f"XGBoost: AUC={auc_xgb:.4f}, время={time_xgb:.2f} с")


In [ ]:
# Проверка XGBoost
assert 'params_xgb' in globals(), "❌ params_xgb не определён."
assert params_xgb.get('n_estimators') == 150, f"❌ n_estimators должно быть 150, у вас {params_xgb.get('n_estimators')}"
assert params_xgb.get('learning_rate') == 0.05, f"❌ learning_rate должно быть 0.05, у вас {params_xgb.get('learning_rate')}"
assert params_xgb.get('max_depth') == 6, f"❌ max_depth должно быть 6, у вас {params_xgb.get('max_depth')}"
assert params_xgb.get('random_state') == 42, f"❌ random_state должно быть 42, у вас {params_xgb.get('random_state')}"
assert 'model_xgb' in globals(), "❌ model_xgb не создана."
assert hasattr(model_xgb, 'fit'), "❌ model_xgb не является моделью (нет метода fit)."
assert 'auc_xgb' in globals(), "❌ auc_xgb не вычислен."
assert 0 <= auc_xgb <= 1, "❌ AUC должен быть в диапазоне [0,1]."
print("✅ XGBoost настроен и обучен верно.")

**Задание 4. LightGBM**

Создайте и обучите LightGBM с параметрами:

*   150 деревьев
*   learning_rate=0.05
*   num_leaves = $2^6$ (так как глубина 6) = 64
*   random_state=42
*   verbose=-1 (отключить вывод)

Замерьте время и ROC-AUC.

In [ ]:
import time

params_lgb = {
    'n_estimators': 150,
    'learning_rate': 0.05,
    'num_leaves': 64,
    'random_state': 42,
    'verbose': -1
}
model_lgb = lgb.LGBMClassifier(**params_lgb)
start = time.time()
model_lgb.fit(X_train, y_train)
time_lgb = time.time() - start
pred_lgb = model_lgb.predict_proba(X_test)[:, 1]
auc_lgb = roc_auc_score(y_test, pred_lgb)
print(f"LightGBM: AUC={auc_lgb:.4f}, время={time_lgb:.2f} с")


In [ ]:
# Проверка LightGBM
assert 'params_lgb' in globals(), "❌ params_lgb не определён."
assert params_lgb.get('n_estimators') == 150, f"❌ n_estimators должно быть 150, у вас {params_lgb.get('n_estimators')}"
assert params_lgb.get('learning_rate') == 0.05, f"❌ learning_rate должно быть 0.05, у вас {params_lgb.get('learning_rate')}"
assert params_lgb.get('num_leaves') == 64, f"❌ num_leaves должно быть 64, у вас {params_lgb.get('num_leaves')}"
assert params_lgb.get('random_state') == 42, f"❌ random_state должно быть 42, у вас {params_lgb.get('random_state')}"
assert 'model_lgb' in globals(), "❌ model_lgb не создана."
assert hasattr(model_lgb, 'fit'), "❌ model_lgb не является моделью."
assert 'auc_lgb' in globals(), "❌ auc_lgb не вычислен."
print("✅ LightGBM настроен и обучен верно.")

**Задание 5. CatBoost**

Обучите CatBoost с параметрами:

*   iterations=150
*   learning_rate=0.05
*   depth=6
*   random_state=42
*   verbose=False

Замерьте время и качество.

In [ ]:
import time

params_cb = {
    'iterations': 150,
    'learning_rate': 0.05,
    'depth': 6,
    'random_state': 42,
    'verbose': False
}
model_cb = cb.CatBoostClassifier(**params_cb)
start = time.time()
model_cb.fit(X_train, y_train)
time_cb = time.time() - start
pred_cb = model_cb.predict_proba(X_test)[:, 1]
auc_cb = roc_auc_score(y_test, pred_cb)
print(f"CatBoost: AUC={auc_cb:.4f}, время={time_cb:.2f} с")


In [ ]:
# Проверка CatBoost
assert 'params_cb' in globals(), "❌ params_cb не определён."
assert params_cb.get('iterations') == 150, f"❌ iterations должно быть 150, у вас {params_cb.get('iterations')}"
assert params_cb.get('learning_rate') == 0.05, f"❌ learning_rate должно быть 0.05, у вас {params_cb.get('learning_rate')}"
assert params_cb.get('depth') == 6, f"❌ depth должно быть 6, у вас {params_cb.get('depth')}"
assert params_cb.get('random_state') == 42, f"❌ random_state должно быть 42, у вас {params_cb.get('random_state')}"
assert 'model_cb' in globals(), "❌ model_cb не создана."
assert hasattr(model_cb, 'fit'), "❌ model_cb не является моделью."
assert 'auc_cb' in globals(), "❌ auc_cb не вычислен."
print("✅ CatBoost настроен и обучен верно.")

**Задание 6. Sklearn GradientBoosting**

Обучите классическую реализацию из sklearn:



*   n_estimators=150
*   learning_rate=0.05
*   max_depth=6
*   random_state=42

Замерьте время и ROC-AUC.

In [ ]:
import time

params_sk = {
    'n_estimators': 150,
    'learning_rate': 0.05,
    'max_depth': 6,
    'random_state': 42
}
model_sk = GradientBoostingClassifier(**params_sk)
start = time.time()
model_sk.fit(X_train, y_train)
time_sk = time.time() - start
pred_sk = model_sk.predict_proba(X_test)[:, 1]
auc_sk = roc_auc_score(y_test, pred_sk)
print(f"Sklearn GBM: AUC={auc_sk:.4f}, время={time_sk:.2f} с")


In [ ]:
# Проверка Sklearn
assert 'params_sk' in globals(), "❌ params_sk не определён."
assert params_sk.get('n_estimators') == 150, f"❌ n_estimators должно быть 150, у вас {params_sk.get('n_estimators')}"
assert params_sk.get('learning_rate') == 0.05, f"❌ learning_rate должно быть 0.05, у вас {params_sk.get('learning_rate')}"
assert params_sk.get('max_depth') == 6, f"❌ max_depth должно быть 6, у вас {params_sk.get('max_depth')}"
assert params_sk.get('random_state') == 42, f"❌ random_state должно быть 42, у вас {params_sk.get('random_state')}"
assert 'model_sk' in globals(), "❌ model_sk не создана."
assert hasattr(model_sk, 'fit'), "❌ model_sk не является моделью."
assert 'auc_sk' in globals(), "❌ auc_sk не вычислен."
print("✅ Sklearn GradientBoosting настроен и обучен верно.")

**Задание 7. Сравнение результатов**

Соберите все результаты в словарь и выведите итоговую таблицу, отсортированную по убыванию AUC.

In [ ]:
results = {
    'XGBoost': {'AUC': auc_xgb, 'Time': time_xgb},
    'LightGBM': {'AUC': auc_lgb, 'Time': time_lgb},
    'CatBoost': {'AUC': auc_cb, 'Time': time_cb},
    'Sklearn': {'AUC': auc_sk, 'Time': time_sk}
}
df_results = pd.DataFrame(results).T
df_results = df_results.sort_values('AUC', ascending=False)
print(df_results.round(4))


In [ ]:
# Проверка словаря результатов
assert 'results' in globals(), "❌ Словарь results не создан."
assert set(results.keys()) == {'XGBoost', 'LightGBM', 'CatBoost', 'Sklearn'}, "❌ В results должны быть все четыре модели."
for name, metrics in results.items():
    assert 'AUC' in metrics, f"❌ Для {name} отсутствует ключ 'AUC'"
    assert 'Time' in metrics, f"❌ Для {name} отсутствует ключ 'Time'"
assert 'df_results' in globals(), "❌ DataFrame df_results не создан."
assert df_results.shape[0] == 4, "❌ В df_results должно быть 4 строки."
print("✅ Результаты собраны корректно.")

**Задание 8. Эксперимент с learning_rate**

Повторите обучение любой из моделей (например, XGBoost) с learning_rate=0.3 и тем же числом деревьев (150).

Как изменилось качество и время? Запишите наблюдения.

In [ ]:
import time

params_xgb_fast = {
    'n_estimators': 150,
    'learning_rate': 0.3,
    'max_depth': 6,
    'random_state': 42,
    'use_label_encoder': False,
    'eval_metric': 'logloss'
}
model_xgb_fast = xgb.XGBClassifier(**params_xgb_fast)
start = time.time()
model_xgb_fast.fit(X_train, y_train)
time_fast = time.time() - start
pred_fast = model_xgb_fast.predict_proba(X_test)[:, 1]
auc_fast = roc_auc_score(y_test, pred_fast)
print(f"XGBoost (lr=0.3): AUC={auc_fast:.4f}, время={time_fast:.2f} с")
print(f"Базовый XGBoost (lr=0.05): AUC={auc_xgb:.4f}, время={time_xgb:.2f} с")


In [ ]:
# Проверка эксперимента
assert 'params_xgb_fast' in globals(), "❌ params_xgb_fast не определён."
assert params_xgb_fast.get('learning_rate') == 0.3, f"❌ learning_rate должно быть 0.3, у вас {params_xgb_fast.get('learning_rate')}"
assert 'model_xgb_fast' in globals(), "❌ model_xgb_fast не создана."
assert 'auc_fast' in globals(), "❌ auc_fast не вычислен."
print("✅ Эксперимент выполнен. Ваши выводы:")
# Здесь мы не можем проверить текстовый вывод, просто напоминаем записать наблюдения.

Ваши наблюдения:

При `learning_rate=0.3` модель делает более крупные шаги на каждой итерации, поэтому при том же числе деревьев обучение обычно проходит чуть быстрее или сопоставимо по времени. Качество может как улучшиться, так и ухудшиться: большой шаг быстрее подгоняет ансамбль под данные, но повышает риск переобучения. Поэтому `learning_rate` лучше подбирать вместе с `n_estimators` на валидации.
